# Hybrid RAG for ERP — notebook roboczy

Komórki są ułożone w kolejności uruchamiania. Sekcje 0–1 robisz raz,
sekcje 2–6 przy każdej zmianie bazy wiedzy.

**Uwaga:** `autoreload` przeładowuje kod z `app/*.py`, ale NIE odświeża wartości
wczytanych z `.env` przy imporcie. Po zmianie `.env` restartuj kernel.


In [ ]:
%load_ext autoreload
%autoreload 2


---
## 0. Instalacja i modele

Jednorazowo. Jeśli `.env` nie istnieje, import `app.core` rzuci czytelny błąd — uzupełnij plik i restartuj kernel.


In [ ]:
!pip install -r requirements.txt
!pip install fastapi uvicorn requests


In [ ]:
from app.core import LLM_MODEL, EMBED_MODEL, GRAPH_MODEL

print(f'{LLM_MODEL=}\n{EMBED_MODEL=}\n{GRAPH_MODEL=}')


In [ ]:
!ollama pull {LLM_MODEL}
!ollama pull {EMBED_MODEL}
!ollama pull {GRAPH_MODEL}


### Baza grafowa (Neo4j + APOC)


In [1]:
!docker compose -f database/docker-compose.yml --env-file .env up -d


 Network erp-assistant-dbs_default Creating 
 Network erp-assistant-dbs_default Created 
 Container erp-assistant-graph-db Creating 
 Container erp-assistant-graph-db Created 
 Container erp-assistant-graph-db Starting 
 Container erp-assistant-graph-db Started 


---
## 1. Sanity check

Jeden import wychwytuje literówki, brakujące zmienne w `.env` i złe ścieżki.


In [2]:
import app.core, app.graph, app.schema, app.ingest, app.plan, app.assistant
from app.graph import PROMPTS_DIR, GRAPHS_DIR
from app.ingest import KNOWLEDGE_DIR

print('importy OK')
print(f'{PROMPTS_DIR}  istnieje={PROMPTS_DIR.exists()}')
print(f'{KNOWLEDGE_DIR}  istnieje={KNOWLEDGE_DIR.exists()}')
print(f'{GRAPHS_DIR}')


importy OK
C:\Users\BKZ\Desktop\Hybrid-RAG-For-ERP-Management\system  istnieje=True
C:\Users\BKZ\Desktop\Hybrid-RAG-For-ERP-Management\knowledge  istnieje=True
C:\Users\BKZ\Desktop\Hybrid-RAG-For-ERP-Management\knowledge\graphs


### Test modeli (szybki)


In [3]:
from app.core import ChatModel, EmbedModel, LLM_MODEL, EMBED_MODEL, EMBED_MODEL_DIM

chat = ChatModel(model=LLM_MODEL, system='Odpowiadasz jednym zdaniem.', memory=False)
print(chat.ask('Powiedz cokolwiek.', think=False))

wektor = EmbedModel(EMBED_MODEL).encode('test')[0]
print(f'wymiar={len(wektor)}, EMBED_MODEL_DIM={EMBED_MODEL_DIM}, zgodne={len(wektor) == EMBED_MODEL_DIM}')


Mogę opisać, jak wygląda zachmurzone niebo nad miastem w porze zmierzchu.
wymiar=1024, EMBED_MODEL_DIM=1024, zgodne=True


---
## 2. Ingest

`purge_database` czyści bazę — węzły sprzed zmiany schematu (bez `modul`, bez `Krok`)
psułyby filtrowanie i plany.


In [4]:
from app import graph

graph.initialize_graph_driver()
graph.purge_database(driver=graph.graph_driver)
print('baza wyczyszczona')


OK: Połączenie z neo4j działa
APOC jest dostępne.
baza wyczyszczona


Poniższa komórka zatrzyma się na `input()` — obejrzyj `print_graph()` powyżej,
sprawdź linię **`Dodano N kroków do M procedur`**, dopiero potem ENTER.

Jeśli `M` < liczby plików w `knowledge/procedures/`, LLM nie trzymał się konwencji nazw.


In [5]:
from app import core
from app.ingest import ingest_llm

ingest_llm(driver=graph.graph_driver, model=core.GRAPH_MODEL)


{'id': 'proc.magazyn.przyjecie-pz', 'title': 'Przyjęcie towaru na magazyn dokumentem PZ', 'module': 'magazyn', 'summary': 'Jak przyjąć towar od dostawcy na wybrany magazyn i zatwierdzić dokument.', 'query': ['jak przyjąć towar', 'towar przyjechał co teraz', 'jak zrobić pz', 'jak zwiększyć stan magazynowy', 'przyjęcie zewnętrzne'], 'preconditions': [], 'roles': ['magazynier', 'kierownik'], 'steps': [{'text': 'Przejdź do Dokumenty w menu bocznym', 'anchor': 'nav.documents', 'action': {'kind': 'click'}}, {'text': 'Kliknij Nowy dokument', 'anchor': 'btn.document-new', 'action': {'kind': 'click'}}, {'text': 'W polu Typ dokumentu wybierz PZ', 'anchor': 'field.document-type', 'action': {'kind': 'select', 'label': 'PZ'}, 'note': 'Po zapisaniu typu nie da się zmienić'}, {'text': 'W polu Dostawca wybierz kontrahenta, od którego przyjmujesz towar', 'anchor': 'field.counterparty', 'action': {'kind': 'select', 'label': 'Stalmex'}}, {'text': 'W sekcji Magazyny wybierz Magazyn docelowy', 'anchor': 'f

Zapisywanie relacji: 100%|██████████| 40/40 [00:00<00:00, 103.32it/s]

{'nodes': 38, 'relations': 40, 'embeddings': 38}

Gotowe!
Możesz podglądać wyniki na: http://localhost:7474


---
## 3. Weryfikacja grafu

Najpierw w pamięci, potem w bazie.


In [6]:
kg = graph.knowledge_graph

print('Klasy:  ', list(kg.classes.keys()))
print('Relacje:', list(kg.relations.keys()))

kroki = [n for n, node in kg.nodes.items() if node.c_name == 'Krok']
print(f'\nWęzły łącznie: {len(kg.nodes)}, w tym Krok: {len(kroki)}')

moduly = {}
for node in kg.nodes.values():
    moduly[node.module] = moduly.get(node.module, 0) + 1
print('Moduły:', moduly)


Klasy:   ['Procedura', 'Blad', 'Koncepcja', 'Krok']
Relacje: ['DOTYCZY', 'WYMAGA_ROZWIAZANIA', 'MA_KROK', 'WYMAGA']

Węzły łącznie: 38, w tym Krok: 27
Moduły: {'magazyn': 38}


Zgodność nazw węzłów z konwencją (`proc.magazyn.pz` → `proc_magazyn_pz`):


In [7]:
from app.ingest import load_knowledge
from app.plan import node_id_from_document_id
from app.schema import Procedure

for d in load_knowledge():
    if isinstance(d, Procedure):
        oczekiwany = node_id_from_document_id(d.id)
        status = 'OK' if oczekiwany in kg.nodes else 'BRAK W GRAFIE'
        print(f'  {d.id:32} -> {oczekiwany:36} {status}')


  proc.magazyn.przyjecie-pz        -> proc_magazyn_przyjecie_pz            OK
  proc.magazyn.wydanie-wz          -> proc_magazyn_wydanie_wz              OK
  proc.magazyn.przesuniecie-mm     -> proc_magazyn_przesuniecie_mm         OK
  proc.magazyn.sprawdzenie-stanu   -> proc_magazyn_sprawdzenie_stanu       OK


In [ ]:
!start http://localhost:7474


W Neo4j Browser (hasło z `.env`):

```cypher
MATCH (n) RETURN labels(n) AS etykiety, count(*) AS ile ORDER BY ile DESC;

MATCH (p)-[r:MA_KROK]->(k) RETURN p.node_id, r.kolejnosc, k.tekst, k.anchor
ORDER BY p.node_id, r.kolejnosc LIMIT 30;

MATCH (a)-[:WYMAGA]->(b) RETURN a.node_id, b.node_id;

MATCH (n) WHERE n.embeddings IS NOT NULL
RETURN n.modul AS modul, count(*) AS z_wektorem ORDER BY modul;
```


---
## 4. Test planu (trawersja)


In [8]:
from app.plan import build_plan

PROCEDURA = 'proc_magazyn_przyjecie_pz'

for row in build_plan(graph.graph_driver, PROCEDURA):
    print(f"[{row['procedura']}] {row['tekst'][:60]!r} | anchor={row['anchor']}")


[proc_magazyn_przyjecie_pz] 'Przejdź do Dokumenty w menu bocznym' | anchor=nav.documents
[proc_magazyn_przyjecie_pz] 'Kliknij Nowy dokument' | anchor=btn.document-new
[proc_magazyn_przyjecie_pz] 'W polu Typ dokumentu wybierz PZ' | anchor=field.document-type
[proc_magazyn_przyjecie_pz] 'W polu Dostawca wybierz kontrahenta, od którego przyjmujesz ' | anchor=field.counterparty
[proc_magazyn_przyjecie_pz] 'W sekcji Magazyny wybierz Magazyn docelowy' | anchor=field.warehouse-to
[proc_magazyn_przyjecie_pz] 'Przejdź na zakładkę Pozycje' | anchor=tab.lines
[proc_magazyn_przyjecie_pz] 'Kliknij Dodaj pozycję' | anchor=btn.line-add
[proc_magazyn_przyjecie_pz] 'Wybierz produkt z listy' | anchor=field.line-product
[proc_magazyn_przyjecie_pz] 'Wpisz przyjmowaną ilość' | anchor=field.line-quantity
[proc_magazyn_przyjecie_pz] 'Kliknij Zatwierdź dokument' | anchor=btn.document-confirm


### Test elastyczności — dodaje relację, pokazuje efekt, **sprząta po sobie**

Bez sprzątania testowa relacja zostaje w bazie i psuje realne odpowiedzi API.


In [9]:
A, B = 'proc_magazyn_wydanie_wz', 'proc_magazyn_przyjecie_pz'

print(f'PRZED: {len(build_plan(graph.graph_driver, A))} kroków')

graph.knowledge_graph.relationship(A, [B], 'WYMAGA', None)
graph.knowledge_graph.sync(driver=graph.graph_driver)
print(f'PO:    {len(build_plan(graph.graph_driver, A))} kroków')

for row in build_plan(graph.graph_driver, A):
    print(f"  [{row['procedura']}] {row['tekst'][:45]!r}")


PRZED: 8 kroków


Zapisywanie relacji: 100%|██████████| 41/41 [00:00<00:00, 272.14it/s]

PO:    18 kroków
  [proc_magazyn_przyjecie_pz] 'Przejdź do Dokumenty w menu bocznym'
  [proc_magazyn_przyjecie_pz] 'Kliknij Nowy dokument'
  [proc_magazyn_przyjecie_pz] 'W polu Typ dokumentu wybierz PZ'
  [proc_magazyn_przyjecie_pz] 'W polu Dostawca wybierz kontrahenta, od które'
  [proc_magazyn_przyjecie_pz] 'W sekcji Magazyny wybierz Magazyn docelowy'
  [proc_magazyn_przyjecie_pz] 'Przejdź na zakładkę Pozycje'
  [proc_magazyn_przyjecie_pz] 'Kliknij Dodaj pozycję'
  [proc_magazyn_przyjecie_pz] 'Wybierz produkt z listy'
  [proc_magazyn_przyjecie_pz] 'Wpisz przyjmowaną ilość'
  [proc_magazyn_przyjecie_pz] 'Kliknij Zatwierdź dokument'
  [proc_magazyn_wydanie_wz] 'Przejdź do Dokumenty i kliknij Nowy dokument'
  [proc_magazyn_wydanie_wz] 'W polu Typ dokumentu wybierz WZ'
  [proc_magazyn_wydanie_wz] 'W polu Odbiorca wybierz kontrahenta, do które'
  [proc_magazyn_wydanie_wz] 'W sekcji Magazyny wybierz Magazyn źródłowy'
  [proc_magazyn_wydanie_wz] 'Przejdź na zakładkę Pozycje'
  [proc_magazyn

In [10]:
# SPRZĄTANIE -- usuwa testową relację z bazy i z pamięci
graph.graph_driver.execute_query(
    'MATCH (a {node_id: $a})-[r:WYMAGA]->(b {node_id: $b}) DELETE r', a=A, b=B)
graph.knowledge_graph.nodes[A].n_relations.pop('WYMAGA', None)

print(f'PO SPRZĄTANIU: {len(build_plan(graph.graph_driver, A))} kroków')


PO SPRZĄTANIU: 8 kroków


---
## 5. Strojenie progu `MIN_SCORE`

Ustaw `ASSISTANT_MIN_SCORE` w `.env` **powyżej** najlepszego trafienia dla pytań
spoza korpusu, ale **poniżej** najgorszego dla pytań sensownych.


In [11]:
graph.initialize_embed_model()

pytania = [
    ('W korpusie', 'jak przyjąć towar na magazyn'),
    ('W korpusie', 'jak wystawić dokument WZ'),
    ('W korpusie', 'jak przesunąć towar między magazynami'),
    ('W korpusie', 'co to jest dokument PZ'),
    ('SPOZA',      'jaka jest stolica Francji'),
    ('SPOZA',      'jak ugotować makaron'),
]

for etykieta, q in pytania:
    wektor = graph.embed_model.encode(q)[0]
    wyniki = graph.KnowledgeGraph.search_semantic(graph.graph_driver, wektor, top_k=3)
    print(f'\n[{etykieta}] {q}')
    for w in wyniki:
        print(f"   {w['score']:.3f}  {w['klasa']:12} {w['node_id']}")



[W korpusie] jak przyjąć towar na magazyn
   0.838  Procedura    proc_magazyn_przyjecie_pz
   0.776  Krok         proc_magazyn_przyjecie_pz__krok_9
   0.776  Procedura    proc_magazyn_przesuniecie_mm

[W korpusie] jak wystawić dokument WZ
   0.843  Krok         proc_magazyn_wydanie_wz__krok_2
   0.803  Krok         proc_magazyn_przyjecie_pz__krok_3
   0.795  Procedura    proc_magazyn_wydanie_wz

[W korpusie] jak przesunąć towar między magazynami
   0.837  Procedura    proc_magazyn_przesuniecie_mm
   0.783  Blad         ERR_1005
   0.780  Blad         ERR_3001

[W korpusie] co to jest dokument PZ
   0.812  Krok         proc_magazyn_przyjecie_pz__krok_3
   0.768  Krok         proc_magazyn_wydanie_wz__krok_2
   0.755  Procedura    proc_magazyn_przyjecie_pz

[SPOZA] jaka jest stolica Francji
   0.638  Krok         proc_magazyn_przyjecie_pz__krok_5
   0.637  Krok         proc_magazyn_przesuniecie_mm__krok_4
   0.623  Procedura    proc_magazyn_sprawdzenie_stanu

[SPOZA] jak ugotować makaron

### Filtr modułu — czy zawężanie działa


In [12]:
wektor = graph.embed_model.encode('jak przyjąć towar')[0]

for modul in [None, 'Magazyn', 'Ksiegowosc']:
    wyniki = graph.KnowledgeGraph.search_semantic(graph.graph_driver, wektor, top_k=5, module=modul)
    print(f'module={modul!r:14} -> {len(wyniki)} wyników')


module=None           -> 5 wyników
module='Magazyn'      -> 5 wyników
module='Ksiegowosc'   -> 0 wyników


---
## 6. Warstwa asystenta (bez HTTP)

Sprawdza `assistant.answer` zanim wejdzie w grę serwer — łatwiej debugować.


In [13]:
from app.assistant import answer
import json

for q in ['jak przyjąć towar na magazyn', 'co to jest dokument PZ', 'jaka jest stolica Francji']:
    odp = answer(q)
    print(f'\n=== {q}')
    print(f"   refused={odp['refused']}, kroków={len(odp['steps'])}, sources={odp['sources']}")
    print(f"   text: {odp['text'][:100]}")



=== jak przyjąć towar na magazyn
   refused=False, kroków=10, sources=['proc_magazyn_przyjecie_pz']
   text: Przyjęcie towaru polega zatwierdzeniu dokumentu i automatycznym zwiększeniu stanu produktów.

=== co to jest dokument PZ
   refused=False, kroków=0, sources=['concept_dokument_magazynowy']
   text: Dokument PZ to rodzaj dokumentu magazynowego służący do przyjęcia towaru z zewnątrz i zwiększenia je

=== jaka jest stolica Francji
   refused=True, kroków=0, sources=[]
   text: Nie znalazłem tego w bazie wiedzy. Spróbuj zapytać inaczej albo skontaktuj się z administratorem sys


---
## 7. API

Serwer uruchom w **osobnym terminalu** (nie w notebooku — zablokuje kernel):

```bash
uvicorn app.api:app --reload --port 8000
```


In [14]:
import requests

print(requests.get('http://localhost:8000/assistant/health').json())


{'ok': True, 'wezly': 38, 'z_embeddingami': 38}


### Asercja kontraktu `AssistantReply`

To jest bramka przed podpięciem frontu — sprawdza kształt, nie treść.


In [15]:
import requests

def sprawdz(pytanie, context=None):
    odp = requests.post('http://localhost:8000/assistant/ask',
                        json={'question': pytanie, 'context': context}).json()

    assert set(odp) == {'text', 'steps', 'sources', 'refused'}, odp.keys()
    assert isinstance(odp['text'], str) and isinstance(odp['refused'], bool)
    assert isinstance(odp['sources'], list)

    for s in odp['steps']:
        assert set(s) <= {'text', 'anchor', 'action', 'note'}, s
        assert isinstance(s['text'], str) and s['text']
        if 'action' in s:
            assert s['action']['kind'] in {'navigate', 'click', 'fill', 'select'}, s['action']

    print(f"OK  refused={odp['refused']}  kroków={len(odp['steps'])}  {pytanie}")
    return odp

sprawdz('jak przyjąć towar na magazyn')
sprawdz('co to jest dokument PZ')
sprawdz('jaka jest stolica Francji')
sprawdz('nie mogę zapisać dokumentu',
        {'module': 'Magazyn', 'lastError': {'code': 'ERR-1004'}})


OK  refused=False  kroków=10  jak przyjąć towar na magazyn
OK  refused=False  kroków=0  co to jest dokument PZ
OK  refused=True  kroków=0  jaka jest stolica Francji
OK  refused=False  kroków=0  nie mogę zapisać dokumentu


{'text': 'Aby rozwiązać błąd zapisywania dokumentu (ERR-1004), należy sprawdzić stan magazynowy w widoku Stany magazynowe. Jeśli ilość na pozycji jest za mała, zmniejsz ją lub najpierw przyjmij brakującą część dokumentem PZ. Błąd występuje, gdy nie ma wystarczającej ilości produktu na magazynie źródłowym do zatwierdzenia dokumentu.',
 'steps': [],
 'sources': ['ERR_1004'],
 'refused': False}

### Podgląd pełnej odpowiedzi (kroki + akcje autopilota)


In [16]:
odp = requests.post('http://localhost:8000/assistant/ask',
                    json={'question': 'jak przyjąć towar na magazyn'}).json()

print(odp['text'], '\n')
for i, s in enumerate(odp['steps'], 1):
    print(f"{i}. {s['text']}")
    print(f"      anchor={s.get('anchor')}  action={s.get('action')}  note={s.get('note')}")


Przyjęcie towaru polega na zatwierdzeniu dokumentu i automatycznym zwiększeniu stanu produktów. 

1. Przejdź do Dokumenty w menu bocznym
      anchor=nav.documents  action={'kind': 'click', 'anchor': 'nav.documents'}  note=None
2. Kliknij Nowy dokument
      anchor=btn.document-new  action={'kind': 'click', 'anchor': 'btn.document-new'}  note=None
3. W polu Typ dokumentu wybierz PZ
      anchor=field.document-type  action={'kind': 'select', 'anchor': 'field.document-type', 'label': 'PZ'}  note=Po zapisaniu typu nie da się zmienić
4. W polu Dostawca wybierz kontrahenta, od którego przyjmujesz towar
      anchor=field.counterparty  action={'kind': 'select', 'anchor': 'field.counterparty', 'label': 'Stalmex'}  note=None
5. W sekcji Magazyny wybierz Magazyn docelowy
      anchor=field.warehouse-to  action={'kind': 'select', 'anchor': 'field.warehouse-to', 'label': 'MAG-GL'}  note=None
6. Przejdź na zakładkę Pozycje
      anchor=tab.lines  action={'kind': 'click', 'anchor': 'tab.lines'}  no

---
## 8. Kopie zapasowe grafu

`ingest_llm` zapisuje automatycznie do `knowledge/graphs/`. Poniżej ręczne wczytanie —
przydaje się, gdy `sync()` padnie i nie chcesz powtarzać pracy modelu.


In [17]:
import os
from app.graph import GRAPHS_DIR

for f in sorted(GRAPHS_DIR.glob('*.json'), reverse=True)[:10]:
    print(f'{f.stat().st_size/1024:8.1f} KB  {f.name}')


    25.1 KB  2026-08-09_13-38-58_qwen3.5-27b.json


In [ ]:
# graph.load_graph(GRAPHS_DIR / 'NAZWA_PLIKU.json')
# print(len(graph.knowledge_graph.nodes), 'węzłów wczytanych')
